# AgentShield: Strands Decider 2B as a first-pass security gatekeeper

1. A deterministic hard policy runs first on every agent action (mandatory checks, never skipped).
2. Then ONE Strands Decider call with 4 typed questions per event fast-allows/blocks evident cases, routes ambiguous or incomplete cases to targeted read-only evidence tools and serious ones to the Amazon Bedrock Nova Pro semantic analyzer, or both.
3. A deterministic resolver enforces conclusive findings; the analyzer returns structured findings and never overrides authorization rules.

**Prerequisites:** a `.env` with the hosted decider key, AWS credentials with Amazon Bedrock access, and `pip install nbclient ipykernel` (or Jupyter).

Outputs below are from a real run on 23 Oct 2026.

Built with the AWS User Group Madurai Strands Decider hackathon kit

In [1]:
import os, sys
os.chdir('..') if os.path.basename(os.getcwd()) == 'notebooks' else None
sys.path.insert(0, os.getcwd())
from dotenv import load_dotenv
load_dotenv('.env')
print('LLM provider :', os.getenv('LLM_PROVIDER', 'bedrock'))
print('LLM model id :', os.getenv('BEDROCK_MODEL_ID', 'us.amazon.nova-pro-v1:0'))
print('Decider mode :', os.getenv('DECIDER_MODE'))

LLM provider : bedrock
LLM model id : us.amazon.nova-pro-v1:0
Decider mode : ugmdu


In [2]:
from agent.agent import HackathonAgent
agent = HackathonAgent()
for evt in ['EVT-101', 'EVT-105', 'EVT-109', 'EVT-114']:
    print('=' * 70)
    print('Event:', evt)
    try:
        r = agent.chat(f'Evaluate {evt}')
        print('latency_ms:', r.get('latency_ms'))
        print('reply:', (r.get('reply') or '')[:400])
        for i, d in enumerate(r.get('decisions') or []):
            probs = {k: round(v, 2) for k, v in d.items() if isinstance(v, float)}
            print(f'  decision {i}: verdict={d.get("verdict")} decider_ms={d.get("decider_ms")} model={d.get("model")}')
            print('    probabilities:', probs)
    except Exception as e:
        print('ERROR:', type(e).__name__, str(e)[:300])

Event: EVT-101


latency_ms: 8631
reply: Event EVT-101 was evaluated and allowed by AgentShield. The decision was made through the fast path where the Strands Decider first pass allowed the action with a high probability of 0.93, despite a moderate probability of high or critical harm at 0.32. AgentShield acted on the decision to allow, confirmed by the final validation step. No additional lookups or analyzers were required, and the acti
  decision 0: verdict=allow decider_ms=5992 model=strands-decider-2b-hobson-v19
    probabilities: {'confidence': 0.93, 'high_risk': 0.32, 'needs_lookup': 0.0, 'needs_analysis': 0.0}
Event: EVT-105


latency_ms: 10030
reply: Event EVT-105 was evaluated and blocked by AgentShield before it ran. The block was due to an attempt to send CONFIDENTIAL or RESTRICTED data to a personal or blocklisted destination, as per rule H3. The Strands Decider first pass chose to investigate further by verifying the recipient, which led to a SUFFICIENT_DENY decision by the resolver, resulting in the action being blocked. The decider call
  decision 0: verdict=investigate: verify recipient then semantic dlp (fetch evidence at 0.35 was not decisive) decider_ms=7515 model=strands-decider-2b-hobson-v19
    probabilities: {'confidence': 0.35, 'high_risk': 0.53, 'needs_lookup': 0.9, 'needs_analysis': 0.74}
Event: EVT-109


latency_ms: 11339
reply: Event EVT-109 was evaluated and blocked by AgentShield before it ran. The block was due to the detection of CONFIDENTIAL_CONTENT by the SEMANTIC_DLP analyzer, which mapped to a DENY decision under rule fm-2026.10. The Strands Decider first pass suggested allowing the action if authorized with a probability of 0.54, but this was not decisive. The SEMANTIC_DLP analyzer confirmed the presence of conf
  decision 0: verdict=investigate: semantic dlp (allow if authorized at 0.54 was not decisive) decider_ms=6519 model=strands-decider-2b-hobson-v19
    probabilities: {'confidence': 0.54, 'high_risk': 0.37, 'needs_lookup': 0.0, 'needs_analysis': 0.39}
Event: EVT-114


latency_ms: 16345
reply: Event EVT-114 was evaluated and blocked by AgentShield before it ran. The block was due to the detection of CONFIDENTIAL_DERIVATION by the SENSITIVE_DERIVATION_REVIEW analyzer, which mapped to a DENY decision under rule fm-2026.10. The Strands Decider first pass chose to investigate further by getting the provenance of the data, but this did not provide a decisive outcome. The SENSITIVE_DERIVATION
  decision 0: verdict=investigate: get provenance then sensitive derivation review (fetch evidence at 0.39 was not decisive) decider_ms=12218 model=strands-decider-2b-hobson-v19
    probabilities: {'confidence': 0.39, 'high_risk': 0.53, 'needs_lookup': 0.86, 'needs_analysis': 0.84}


In [3]:
print(open('submission/bench/results.md').read())

Decider: strands-decider-2b-hobson-v19 (ugmdu). 15 labelled cases, 54 decisions.

| | Strands Decider 2B | us.amazon.nova-pro-v1:0 |
|---|---|---|
| Accuracy | 61.1% | 61.1% |
| Latency p50 per decision | 6824 ms | 2085 ms |
| Latency p95 | 9937 ms | 2656 ms |
| Cost per 1,000 decisions | $0.0350 | $0.6003 |
| Answered alone at confidence >= 0.7 | 7% | 100% |
| Accuracy when confident | 100.0% | 61.1% |

At the median the LLM was 3.2x faster than the decider on this setup (ugmdu).



In [4]:
import urllib.parse, subprocess
runtime_arn = 'arn:aws:bedrock-agentcore:us-east-1:253270662483:runtime/deciderkit_agent-QSkyjVEnov'
url = ('https://bedrock-agentcore.us-east-1.amazonaws.com/runtimes/'
       + urllib.parse.quote(runtime_arn, safe='') + '/invocations?qualifier=DEFAULT')
print('Deployed endpoint (IAM SigV4-authenticated):')
print(url)
try:
    p = subprocess.run(['./kit', 'invoke', 'Evaluate EVT-105'], capture_output=True, text=True, timeout=180)
    print('exit code:', p.returncode)
    print(p.stdout[-2500:])
    if p.stderr.strip():
        print('stderr tail:', p.stderr[-800:])
except Exception as e:
    print('ERROR:', type(e).__name__, str(e)[:300])

Deployed endpoint (IAM SigV4-authenticated):
https://bedrock-agentcore.us-east-1.amazonaws.com/runtimes/arn%3Aaws%3Abedrock-agentcore%3Aus-east-1%3A253270662483%3Aruntime%2Fdeciderkit_agent-QSkyjVEnov/invocations?qualifier=DEFAULT


exit code: 0
arm 0.53; AgentShield acted on: investigate: verify recipient then semantic dlp (fetch evidence at 0.35 was not decisive) (low-confidence rule)",
    "VERIFY_RECIPIENT -> resolver SUFFICIENT_DENY"
   ],
   "decider_calls": 1,
   "lookups": [
    "VERIFY_RECIPIENT"
   ],
   "analyzers": [],
   "prevented": true,
   "approval_request": null,
   "advisory": [],
   "note": "already evaluated in this chat; stored outcome (no new decision)"
  }
 ],
 "approvals": [],
 "trace": [
  {
   "event_id": "EVT-105",
   "trace": [
    {
     "stage": "resolver",
     "after": "VERIFY_RECIPIENT",
     "status": "SUFFICIENT_DENY",
     "rule": "H3",
     "reason": "CONFIDENTIAL or RESTRICTED data to a personal or blocklisted destination"
    }
   ],
   "stage_ms": {
    "policy": 0.159,
    "routing": 7040.366,
    "evidence": 0.042,
    "resolver": 0.018,
    "enforcement": 0.012
   },
   "routing": [
    {
     "event_id": "EVT-105",
     "step_index": 0,
     "eligible_options": [
      

## Links

- Repository: https://github.com/nengxu2008/agentshield-strands-decider
- Demo video: https://youtu.be/tGztQFYy4Eg